# Day 5.5 — Events, Logs and Checkpoints
Two records that are easy to confuse. **Events** are append-only observations, never edited. A
**checkpoint** is mutable continuation state, deleted the moment it is used. Here both become
durable, and the retry budget from 5.2 gets something to survive.

### Why the two records cannot be merged

An event log is an audit record and does not promise to hold everything needed to continue; a
checkpoint is not an audit log and gets deleted once used. Durable approval needs the exact pending
action under a stable run id, and sensitive arguments should be redacted from telemetry.

**Cost** belongs to a whole run, retries included — which is why usage rides on every model event
rather than on the final result. Mock mode reports zeros honestly: nothing was bought.

### Step 1 — A fresh run folder, and events on disk

Give `EventStore` a path and every event lands in a JSON-lines file as it happens.

In [ ]:
RUN_ROOT = WORKSPACE / "runs" / f"events_{datetime.now():%Y%m%d_%H%M%S}"
RUN_ROOT.mkdir(parents=True, exist_ok=True)

class JSONCheckpointStore:
    """The same save/load/delete interface, backed by one JSON file per run."""
    def __init__(self, directory):
        self.directory = Path(directory)
        self.directory.mkdir(parents=True, exist_ok=True)
    def _path(self, run_id): return self.directory / f"{run_id}.json"
    def save(self, run_id, state): self._path(run_id).write_text(json.dumps(state, indent=2), encoding="utf-8")
    def load(self, run_id):
        path = self._path(run_id)
        return json.loads(path.read_text(encoding="utf-8")) if path.exists() else None
    def delete(self, run_id):
        if self._path(run_id).exists(): self._path(run_id).unlink()

durable_events = EventStore(RUN_ROOT / "events.jsonl")
checkpoints = JSONCheckpointStore(RUN_ROOT / "checkpoints")
runtime = HarnessRuntime(build_demo_registry(), MockProvider(), durable_events, checkpoints)

paused = runtime.run(task, "Send the synthetic update")
print("Run folder:", RUN_ROOT.name, "| status:", paused.status)
print()
for event in durable_events.get(paused.run_id):
    print(f"  {event['event']:<20}", event["details"])

lines = (RUN_ROOT / "events.jsonl").read_text(encoding="utf-8").splitlines()
print()
print("Lines already written to events.jsonl:", len(lines))
print("First line, raw:", lines[0][:110], "...")
print("Parsed back    :", json.loads(lines[0])["event"], "at", json.loads(lines[0])["timestamp"])

### Step 2 — Restart, and resume from disk

Throw the runtime away and build a new one sharing only the folder. The approval must survive that;
an event log alone could not carry it.

In [ ]:
state = checkpoints.load(paused.run_id)
print("Checkpoint holds:")
for key in sorted(state):
    print(f"  {key:<12}:", f"<{len(state[key])} history entries>" if key == "history" else state[key])
print("On disk as      :", f"{paused.run_id}.json")

# Simulate a restart: a different runtime object, sharing only what is on disk.
restarted = HarnessRuntime(build_demo_registry(), MockProvider(),
                           durable_events, JSONCheckpointStore(RUN_ROOT / "checkpoints"))
print()
print("Brand-new runtime object :", id(restarted) != id(runtime))
resumed = restarted.resume(paused.run_id, task, approved=True)
print("Resumed status           :", resumed.status)
print("Tool output              :", resumed.output)
print("Checkpoint afterwards    :", checkpoints.load(paused.run_id), "(consumed)")
print("Trace across BOTH runtimes:", [e["event"] for e in durable_events.get(paused.run_id)])

### Step 3 — A transient failure is retried, with backoff

`FlakyModel` raises on its first call, then behaves normally. Watch the runtime absorb that using the
budget defined in 5.2.

In [ ]:
class FlakyModel:
    """Teaching double: fails its first `failures` calls, then delegates to a normal provider."""
    def __init__(self, failures=1, inner=None, error="Simulated 429 rate limit from the provider"):
        self.failures, self.inner, self.error, self.calls = failures, inner or MockProvider(), error, 0
    def decide(self, prompt, config, tools, history):
        self.calls += 1
        if self.calls <= self.failures:
            raise RuntimeError(f"{self.error} (attempt {self.calls})")
        return self.inner.decide(prompt, config, tools, history)

recovered = HarnessRuntime(build_demo_registry(), FlakyModel(failures=1), EventStore()).run(research, "harness")

print("Final status:", recovered.status)
for event in recovered.events:
    if event["event"].startswith("provider") or event["event"] in {"model_completed", "run_completed"}:
        print(f"  {event['event']:<28}", event["details"])
print()
print("retry_in_seconds doubles each attempt: 0.05s, then 0.10s. That is exponential backoff.")
print("Production code adds a small random 'jitter' so that clients recovering from one outage")
print("do not all retry at the same instant.")

### Step 4 — The budget is bounded, and some errors are never retried

Retrying forever is a slower outage; retrying the *wrong* error spends credit on something that
cannot improve.

In [ ]:
gave_up = HarnessRuntime(build_demo_registry(), FlakyModel(failures=99), EventStore()).run(research, "harness")
print("Always-failing provider ->", gave_up.status)
print("  attempts recorded     :", len([e for e in gave_up.events if e["event"] == "provider_retry"]) + 1)
for event in gave_up.events:
    if event["event"].startswith(("provider", "run_failed")):
        print(f"    {event['event']:<34}", str(event["details"].get("error", ""))[:64])

class MisconfiguredModel:
    """A ValueError means bad arguments or bad configuration. Repeating it changes nothing."""
    def decide(self, prompt, config, tools, history):
        raise ValueError("model name is not valid for this provider")

not_retried = HarnessRuntime(build_demo_registry(), MisconfiguredModel(), EventStore()).run(research, "hello")
print()
print("Non-transient error ->", not_retried.status)
print("  events            :", [e["event"] for e in not_retried.events])
print("  -> it stops at once instead of burning the budget on an error that cannot succeed.")

### Step 5 — Totalling the bill for a run

Three lines, identical on a live run — failed attempts included.

In [ ]:
totals = {"prompt_tokens": 0, "completion_tokens": 0, "cost_usd": 0.0}
model_calls = [e for e in recovered.events if e["event"] == "model_completed"]
for event in model_calls:
    for key in totals:
        totals[key] += event["details"]["usage"].get(key, 0)

print("Model calls in that run:", len(model_calls))
print("Retries in that run    :", len([e for e in recovered.events if e["event"] == "provider_retry"]))
print("Totals for run", recovered.run_id, ":", totals)
print()
print("Zero, because this was MOCK mode - nothing was bought. In LIVE mode the same three lines")
print("give you a per-run bill, attributed to the whole run rather than to one reply.")

### Try it yourself

Predict: if the checkpoint file is deleted before you resume, what does `resume()` do?

In [ ]:
# --- Worked solution ---------------------------------------------------------------
victim = runtime.run(task, "Send another synthetic update")
path = RUN_ROOT / "checkpoints" / f"{victim.run_id}.json"
print("Paused         :", victim.status, "| checkpoint file exists:", path.exists())
path.unlink()                                     # simulate losing the durable state
print("Deleted it     : exists:", path.exists())

outcome = runtime.resume(victim.run_id, task, approved=True)
print()
print("Status:", outcome.status, "|", outcome.output)
print("It fails closed. Without the exact pending action there is nothing safe to run, and the")
print("harness will not ask the model to invent it again.")
print("The EVENT log still shows the pause:", [e["event"] for e in durable_events.get(victim.run_id)])
print("Events explain; checkpoints continue. That is why they are two different records.")

### Checkpoint

**1. You have the complete event log for a paused run. Can you resume it?**

<details><summary>Show answer</summary>

No. The log explains what happened; it does not promise to hold what is needed to continue. Resuming needs the checkpoint: exact tool, arguments and history, under a stable run id.

</details>

**2. Why does the harness retry a `RuntimeError` but not a `ValueError`?**

<details><summary>Show answer</summary>

A `RuntimeError` means the call failed - a timeout, a rate limit - and may succeed later. A `ValueError` means the request was malformed; repeating it produces the identical error.

</details>

### Recap

- **Limitation seen:** printed output dies with the kernel, and a paused run had nothing durable to resume from.
- **Layer added:** a JSONL event log, a JSON checkpoint store, retries visible as events.
- **Evidence:** a new runtime resumed from disk; one failure recovered; a broken provider gave up on budget.